# 01 — OPEX Data Preparation & Exploration

Line-by-line exploration of the v2.2 SAP-inspired Pharmaceutical OPEX dataset (`seed = 42`):
1. Load raw CSV tables cleanly using `opex_forecast.config` & `opex_forecast.io` (preserving leading zeros and `""` blank keys).
   - Tip: see `00_sap_column_reference.ipynb` or pass `readable=True` to `load_pharma_table(...)` for human-readable `UPPER_SNAKE` column names.
2. Inspect dimension hierarchies (`dim_company_code`, `dim_plant`, `dim_cost_center`, `dim_profit_center`, `dim_gl_account`).
3. Apply the canonical 4-condition recurring OPEX filter on `acdoca_universal_journal`.
4. Inspect `fact_opex_drivers`, `plan_drivers`, and point-in-time `fact_po_commitments`.

In [ ]:
import pandas as pd

from opex_forecast.config import DOCS_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR
from opex_forecast.io import load_pharma_table

print(f"RAW_DATA_DIR: {RAW_DATA_DIR} (exists={RAW_DATA_DIR.exists()})")
print(f"DOCS_DIR:     {DOCS_DIR}")

In [ ]:
dim_gl = load_pharma_table("dim_gl_account")
dim_cc = load_pharma_table("dim_cost_center")
uj = load_pharma_table("acdoca_universal_journal")

# Optional readable view for interactive exploration (see 00_sap_column_reference.ipynb)
uj_readable = load_pharma_table("acdoca_universal_journal", readable=True)
print(f"Readable sample columns: {list(uj_readable.columns[:8])}")

print(f"dim_gl_account:           {dim_gl.shape}")
print(f"dim_cost_center:          {dim_cc.shape}")
print(f"acdoca_universal_journal: {uj.shape}")
dim_gl.head()

In [ ]:
opex_gls = set(dim_gl.loc[dim_gl["ACCOUNT_TYPE"] == "OPEX", "HKONT"])
snapshot_date = "2025-12-31"

clean_uj = uj[
    (uj["RACCT"].isin(opex_gls))
    & (uj["POPER"] <= 12)
    & (uj["SPECIAL_ITEM_FLAG"] == "NONE")
    & (uj["CPUDT"] <= snapshot_date)
]

panel = clean_uj.groupby(
    ["RYEAR", "POPER", "RBUKRS", "WERKS", "RCNTR", "RACCT"], as_index=False
)[["HSL", "KSL", "KSL_BUDGET_FX"]].sum()

print(f"Clean recurring OPEX lines: {len(clean_uj):,} -> Monthly panel rows: {len(panel):,}")
panel.head()